# Prompt Testing Workbench

Use this notebook to:
1. Load a prompt from YAML
2. Inspect its variables
3. Run it with test inputs
4. Modify the prompt (create v2)
5. Run again and compare outputs

In [ ]:
import sys
sys.path.insert(0, "..")

from utils.prompt_loader import load_prompt, fill_prompt, get_prompt_variables, list_prompt_keys, list_all_prompts
from utils.api_client import call_model
from utils.prompt_runner import run_prompt, run_test_cases, compare_versions, save_results

## 0. Fetch Real Inputs from Mangowater

Run a query on the live deployment first, then fetch the exact inputs each prompt received.
These become your test fixtures — real data, no guessing.

In [ ]:
import requests
import json

# Step 1: Set the mangowater DeepSearch URL
DEEPSEARCH_URL = "https://mangowater.norizon.dev"  # <-- adjust if different

# Step 2: Fetch recent prompt logs (after you've run a query on the platform)
response = requests.get(f"{DEEPSEARCH_URL}/api/v1/debug/prompt-logs", params={"limit": 50})
logs = response.json()

print(f"Found {logs['count']} prompt log entries\n")

# Step 3: Browse what's available
for i, entry in enumerate(logs["logs"]):
    print(f"[{i}] {entry['category']}/{entry['prompt_key']}  ({entry['rendered_length']} chars)  {entry['timestamp_iso']}")
    print(f"     Variables: {list(entry['variables'].keys())}")
    print()

In [ ]:
# Step 4: Pick a log entry and use its real variables
LOG_INDEX = 0  # <-- change this to pick a different entry

entry = logs["logs"][LOG_INDEX]
print(f"Using: {entry['category']}/{entry['prompt_key']}")
print(f"Timestamp: {entry['timestamp_iso']}")
print(f"\nVariables captured:")
for k, v in entry["variables"].items():
    preview = v[:200] + "..." if len(v) > 200 else v
    print(f"  {k}: {preview}")

# These are now ready to use in sections below
real_variables = entry["variables"]
real_category = entry["category"]
real_prompt_key = entry["prompt_key"]

## 1. Explore Available Prompts

In [ ]:
# List all prompt files and their keys
all_prompts = list_all_prompts("../prompts")
for file, keys in all_prompts.items():
    print(f"\n📄 {file}")
    for k in keys:
        print(f"   - {k}")

## 2. Load a Specific Prompt

In [ ]:
# Choose the prompt file and key you want to test
PROMPT_FILE = "../prompts/deepsearch/supervisor.yaml"
PROMPT_KEY = "generate_report_factual"

prompt = load_prompt(PROMPT_FILE)
template = prompt[PROMPT_KEY]

print(f"Prompt key: {PROMPT_KEY}")
print(f"Variables needed: {get_prompt_variables(template)}")
print(f"Length: {len(template)} chars")
print("---")
print(template[:500] + "..." if len(template) > 500 else template)

## 3. Run Baseline (v1)

In [ ]:
# Fill in the variables for your test
variables = {
    "query": "Welche Richtlinien gelten für Biegeteile?",
    "search_findings": "Source 1 (Page: Richtlinien für Biegeteile):\nMindestbiegeradius: 1x Materialstärke bei S235.",
    "sources_count": "1",
}

# Preview the filled prompt
filled = fill_prompt(template, variables)
print("=== FILLED PROMPT ===")
print(filled)

In [ ]:
# Run v1
result_v1 = run_prompt(prompt, variables, prompt_key=PROMPT_KEY)

print(f"Duration: {result_v1['duration_seconds']}s")
print("=== OUTPUT v1 ===")
print(result_v1["output"])

## 4. Create v2 (Modified Prompt)

Copy the YAML file, make your changes, and save as a new version.
For example: `prompts/deepsearch/supervisor_v2.yaml`

In [ ]:
# Load the modified prompt
PROMPT_FILE_V2 = "../prompts/deepsearch/supervisor_v2.yaml"  # <-- create this file with your changes

prompt_v2 = load_prompt(PROMPT_FILE_V2)
template_v2 = prompt_v2[PROMPT_KEY]

print(f"v2 length: {len(template_v2)} chars")
print(f"v2 variables: {get_prompt_variables(template_v2)}")

In [ ]:
# Run v2 with the same variables
result_v2 = run_prompt(prompt_v2, variables, prompt_key=PROMPT_KEY)

print(f"Duration: {result_v2['duration_seconds']}s")
print("=== OUTPUT v2 ===")
print(result_v2["output"])

## 5. Compare Outputs

In [ ]:
print("=" * 80)
print("INPUT")
print("=" * 80)
for k, v in variables.items():
    preview = str(v)[:150] + "..." if len(str(v)) > 150 else str(v)
    print(f"  {k}: {preview}")

print("\n" + "=" * 80)
print(f"OUTPUT v1 (baseline) — {result_v1['duration_seconds']}s")
print("=" * 80)
print(result_v1["output"])

print("\n" + "=" * 80)
print(f"OUTPUT v2 (improved) — {result_v2['duration_seconds']}s")
print("=" * 80)
print(result_v2["output"])

## 6. Batch Run (once you have test cases)

After documenting test cases in Excel and exporting to JSON, use this section.

In [ ]:
import json

# Load test cases from JSON (export from your Excel file)
# with open("../test_cases/your_test_cases.json") as f:
#     test_cases = json.load(f)

# results_v1 = run_test_cases(prompt, test_cases, prompt_key=PROMPT_KEY)
# results_v2 = run_test_cases(prompt_v2, test_cases, prompt_key=PROMPT_KEY)

# comparison = compare_versions(results_v1, results_v2)
# print(comparison)

# Save results
# save_results(results_v1, "../results/v1_results.json")
# save_results(results_v2, "../results/v2_results.json")